# 지역별고용조사 기반 청년층 정규직 근로자 비율 산출

2016–2024년 지역별고용조사 C형 상·하반기 자료를 각각 독립된 시점으로 사용해 시도별 청년층 정규직 근로자 비율을 산출한다.

- 분모: 만 19–34세, 경제활동상태 코드 1, 종사상지위 코드 1·2
- 분자: 분모 중 종사상지위 코드 1이고 주업·부업 총계 근무시간 구분 코드 3
- 산식: 분자 시도가중값 합계 ÷ 분모 시도가중값 합계 × 100
- 파일별 시도 비율을 먼저 산출하고, 최종 출력에서만 소수점 첫째 자리로 반올림한다.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = REPO_ROOT / "data/raw/구조환경지수 원데이터 구축용/지역별고용조사"
OUTPUT_DIR = REPO_ROOT / "data/processed/analysis"
OUTPUT_PATH = OUTPUT_DIR / "2016-2024_지역별고용조사_청년층_정규직_근로자_비율.csv"

SIDO_NAMES = {
    11: "서울", 21: "부산", 22: "대구", 23: "인천", 24: "광주",
    25: "대전", 26: "울산", 29: "세종", 31: "경기", 32: "강원",
    33: "충북", 34: "충남", 35: "전북", 36: "전남", 37: "경북",
    38: "경남", 39: "제주",
}
HALF_ORDER = {"상반기": 0, "하반기": 1}

In [2]:
file_records = []
for path in RAW_DIR.glob("*.csv"):
    match = re.match(r"^(20\d{2})_(상반기|하반기)\(C형", path.name)
    if match:
        year, half = match.groups()
        file_records.append({"연도": int(year), "반기": half, "시점": f"{year}_{half}", "경로": path})

file_index = (
    pd.DataFrame(file_records)
    .assign(반기순서=lambda x: x["반기"].map(HALF_ORDER))
    .sort_values(["연도", "반기순서"])
    .drop(columns="반기순서")
    .reset_index(drop=True)
)
assert len(file_index) == 18
file_index[["연도", "반기", "시점", "경로"]]

,연도,반기,시점,경로
0,2016,상반기,2016_상반기,D:\University\yumocha\yumocha\data\raw\구조환경지수 ...
1,2016,하반기,2016_하반기,D:\University\yumocha\yumocha\data\raw\구조환경지수 ...
2,2017,상반기,2017_상반기,D:\University\yumocha\yumocha\data\raw\구조환경지수 ...
3,2017,하반기,2017_하반기,D:\University\yumocha\yumocha\data\raw\구조환경지수 ...
4,2018,상반기,2018_상반기,D:\University\yumocha\yumocha\data\raw\구조환경지수 ...
5,2018,하반기,2018_하반기,D:\University\yumocha\yumocha\data\raw\구조환경지수 ...
6,2019,상반기,2019_상반기,D:\University\yumocha\yumocha\data\raw\구조환경지수 ...
7,2019,하반기,2019_하반기,D:\University\yumocha\yumocha\data\raw\구조환경지수 ...
8,2020,상반기,2020_상반기,D:\University\yumocha\yumocha\data\raw\구조환경지수 ...
9,2020,하반기,2020_하반기,D:\University\yumocha\yumocha\data\raw\구조환경지수 ...


In [3]:
COLUMN_ALIASES = {
    "시도코드": ["행정구역시도코드", "2자리_행정구역시도코드"],
    "만연령": ["만연령"],
    "경제활동상태": ["경제활동인구상태코드", "경제활동구분코드"],
    "종사상지위": ["종사상지위코드", "현직장종사상지위코드"],
    "근무시간구분": ["주업부업총계시간구분코드"],
    "가중치": ["시도전국가중값", "시도가중값"],
}
COMMON_COLUMNS = list(COLUMN_ALIASES)

In [4]:
def calculate_sido_rate(path):
    data = pd.read_csv(path, encoding="cp949")
    rename_map = {
        next(alias for alias in aliases if alias in data.columns): common
        for common, aliases in COLUMN_ALIASES.items()
    }
    data = data.rename(columns=rename_map)[COMMON_COLUMNS]

    denominator = (
        data["만연령"].between(19, 34)
        & data["경제활동상태"].eq(1)
        & data["종사상지위"].isin([1, 2])
    )
    numerator = (
        denominator
        & data["종사상지위"].eq(1)
        & data["근무시간구분"].eq(3)
    )

    denominator_weight = data.loc[denominator].groupby("시도코드")["가중치"].sum()
    numerator_weight = data.loc[numerator].groupby("시도코드")["가중치"].sum()
    return numerator_weight.div(denominator_weight).mul(100)

period_results = {
    row.시점: calculate_sido_rate(row.경로)
    for row in file_index.itertuples(index=False)
}
raw_result = pd.DataFrame(period_results).reindex(SIDO_NAMES)
raw_result.index.name = "시도코드"

In [5]:
final_df = raw_result.rename(index=SIDO_NAMES).round(1).reset_index(names="시도")
assert final_df.shape == (17, 19)
final_df

,시도,2016_상반기,2016_하반기,2017_상반기,2017_하반기,2018_상반기,2018_하반기,2019_상반기,2019_하반기,2020_상반기,2020_하반기,2021_상반기,2021_하반기,2022_상반기,2022_하반기,2023_상반기,2023_하반기,2024_상반기,2024_하반기
0,서울,58.5,65.1,68.3,64.7,67.6,65.4,67.6,69.3,45.4,67.9,69.5,52.5,70.6,51.5,74.6,72.5,72.3,71.7
1,부산,58.6,63.8,65.8,65.4,66.8,68.3,69.2,66.4,47.2,65.7,69.6,63.3,66.7,31.0,65.2,64.0,67.2,68.3
2,대구,60.4,63.2,68.0,66.2,63.8,63.6,67.0,65.2,53.1,66.2,66.4,61.1,69.3,56.6,70.6,69.5,72.5,72.4
3,인천,59.1,62.7,63.5,63.8,66.4,63.5,61.4,62.7,44.0,63.0,66.0,64.9,70.0,58.9,70.9,72.8,72.1,69.7
4,광주,64.6,65.8,64.8,62.6,62.6,63.0,66.7,65.4,53.4,62.6,65.5,63.5,67.3,58.7,68.8,66.5,68.3,68.7
5,대전,59.4,66.3,70.3,69.1,68.1,64.7,72.1,67.0,53.2,63.3,66.8,48.6,70.0,43.0,74.7,74.9,70.8,69.2
6,울산,68.3,73.1,71.3,70.6,69.4,67.9,72.7,68.4,53.2,66.8,76.6,63.9,73.8,63.9,73.5,73.7,70.6,68.9
7,세종,NaN,NaN,73.5,71.7,72.8,71.7,74.6,68.1,32.9,78.0,80.4,58.9,75.4,53.7,75.4,76.2,72.6,69.6
8,경기,61.7,68.5,69.3,67.9,69.4,69.5,68.0,67.9,45.3,68.9,70.1,56.5,70.1,50.3,72.9,70.8,70.6,69.7
9,강원,52.8,66.8,68.6,66.5,67.6,65.7,66.6,67.5,39.3,65.7,64.1,49.1,68.3,42.3,68.6,68.1,70.5,70.5


In [6]:
official_jeju_2024_h1 = 65.4
jeju_2024_h1_raw = raw_result.loc[39, "2024_상반기"]
jeju_2024_h1_rounded = round(jeju_2024_h1_raw, 1)
jeju_2024_h2_raw = raw_result.loc[39, "2024_하반기"]
jeju_2024_h2_rounded = round(jeju_2024_h2_raw, 1)

jeju_comparison = pd.DataFrame(
    {
        "값": [
            official_jeju_2024_h1,
            jeju_2024_h1_raw,
            jeju_2024_h1_rounded,
            jeju_2024_h1_raw - official_jeju_2024_h1,
            jeju_2024_h1_rounded - official_jeju_2024_h1,
            jeju_2024_h2_raw,
            jeju_2024_h2_rounded,
        ]
    },
    index=[
        "2024년 상반기 공식값",
        "2024년 상반기 계산값(반올림 전)",
        "2024년 상반기 계산값(소수점 첫째 자리)",
        "2024년 상반기 차이(반올림 전)",
        "2024년 상반기 차이(소수점 첫째 자리)",
        "2024년 하반기 계산값(반올림 전)",
        "2024년 하반기 계산값(소수점 첫째 자리)",
    ],
)
jeju_comparison

,값
2024년 상반기 공식값,65.400000
2024년 상반기 계산값(반올림 전),65.477988
2024년 상반기 계산값(소수점 첫째 자리),65.500000
2024년 상반기 차이(반올림 전),0.077988
2024년 상반기 차이(소수점 첫째 자리),0.100000
2024년 하반기 계산값(반올림 전),63.589094
2024년 하반기 계산값(소수점 첫째 자리),63.600000


### 제주 공식값 비교 결과

종사상지위 코드 3인 비임금근로자를 분모에서 제외한 수정 산식의 2024년 상반기 제주 값은 반올림 전 65.4780%이고 소수점 첫째 자리 반올림값은 65.5%이다. 공식값 65.4%보다 0.1%p 높으며, 이 차이는 별도 원인 확인 대상으로 남긴다. 계산값을 공식값에 맞춰 조정하지 않았고 비임금근로자 코드 3도 분모에 포함하지 않았다.

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
final_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

reloaded = pd.read_csv(OUTPUT_PATH, encoding="utf-8-sig")
expected_columns = ["시도", *file_index["시점"].tolist()]
assert reloaded.shape == (17, 19)
assert reloaded.columns.tolist() == expected_columns
assert reloaded["시도"].tolist() == list(SIDO_NAMES.values())
assert reloaded.isna().equals(final_df.isna())

print(f"저장 경로: {OUTPUT_PATH.relative_to(REPO_ROOT)}")
print(f"최종 크기: {reloaded.shape[0]}개 시도 × {reloaded.shape[1] - 1}개 반기 값")
print(f"2016년 상반기 결측 시도 수: {reloaded['2016_상반기'].isna().sum()}")

저장 경로: data\processed\analysis\2016-2024_지역별고용조사_청년층_정규직_근로자_비율.csv
최종 크기: 17개 시도 × 18개 반기 값
2016년 상반기 결측 시도 수: 1
